# Test accuracy for every saved model


In [30]:
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import datasets

from cnn import build_model
from train import build_transforms, get_device
from evaluate import evaluate

PROJECT_ROOT = Path('..').resolve()
MODELS_DIR = PROJECT_ROOT / 'Models'
TEST_ROOT = PROJECT_ROOT / 'data' / 'test'
device = get_device()
print('Device:', device)

Device: mps


In [33]:
results = []
for ckpt_path in sorted(MODELS_DIR.glob('*.pt')):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg = ckpt['config']
    cfg['model']['pretrained'] = False

    _, eval_transform = build_transforms(cfg['data'])
    test_dataset = datasets.ImageFolder(TEST_ROOT, transform=eval_transform)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    model = build_model(cfg, num_classes=len(ckpt['class_names']))
    model.load_state_dict(ckpt['model_state'])
    model.to(device)

    _, test_acc = evaluate(model, test_loader, device)
    _, test_acc_tta = evaluate(model, test_loader, device, tta=True)
    print(f'{ckpt_path.name:40s} test acc {test_acc:.4f} | flip-TTA {test_acc_tta:.4f}')
    results.append({'model': ckpt_path.stem, 'best_val_acc': ckpt.get('best_val_acc'),
                    'test_acc': test_acc, 'test_acc_tta': test_acc_tta})

results_df = pd.DataFrame(results).sort_values('test_acc', ascending=False).reset_index(drop=True)
results_df

best_efficientnet_b0_baseline.pt         test acc 0.9175 | flip-TTA 0.9175
efficientnet_b0_best.pt                  test acc 0.9500 | flip-TTA 0.9450
efficientnet_b0_cutmix+erasing.pt        test acc 0.9100 | flip-TTA 0.9075
efficientnet_b0_cutmix.pt                test acc 0.9100 | flip-TTA 0.9075
efficientnet_b0_heavy.pt                 test acc 0.9250 | flip-TTA 0.9425
efficientnet_b0_light.pt                 test acc 0.9175 | flip-TTA 0.9150


,model,best_val_acc,test_acc,test_acc_tta
0,efficientnet_b0_best,0.956250,0.9500,0.9450
1,efficientnet_b0_heavy,0.943750,0.9250,0.9425
2,best_efficientnet_b0_baseline,0.916667,0.9175,0.9175
3,efficientnet_b0_light,0.922917,0.9175,0.9150
4,efficientnet_b0_cutmix+erasing,0.954167,0.9100,0.9075
5,efficientnet_b0_cutmix,0.954167,0.9100,0.9075
